# Email Workflow Agent
 a structured customer email processing agent. This workflow showcases:
- **State**: Tracking email content, sender, draft replies, and a checklist of tasks.
- **Nodes**: Analytical, classification, routing, and operational nodes.
- **Conditional Routing**: Sending the email to specialized handlers (billing, technical, feedback, general) depending on category.
- **Task Tracking**: Task statuses (`pending`, `in_progress`, `completed`).
- **Memory**: Maintaining history for human review and multi-step tasks.
- **Interrupts**: Pausing the workflow for a human agent to inspect/modify draft replies and update task statuses.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
print("OpenAI API Key set:", "OPENAI_API_KEY" in os.environ)
print("LangSmith tracing set:", os.environ.get("LANGCHAIN_TRACING_V2"))

OpenAI API Key set: True
LangSmith tracing set: true


### 1. Define State and Schema
We need to track details about the email, the classification category, task list, and the reply draft.

In [2]:
from typing import TypedDict, List, Dict, Any

class Task(TypedDict):
    description: str
    status: str  # 'pending', 'in_progress', 'completed'

class EmailState(TypedDict):
    email_id: str
    sender: str
    subject: str
    body: str
    category: str  # 'billing', 'technical', 'feedback', 'general'
    tasks: List[Task]
    reply_draft: str
    approved: bool

### 2. Define Workflow Nodes
We'll define nodes using the OpenAI LLM (or simple business logic for demonstration) to categorize emails and generate response drafts.

In [3]:
from mock_llm import get_llm
from langchain_core.prompts import ChatPromptTemplate
import json

llm = get_llm(model="gpt-4o-mini", temperature=0)

def categorize_and_analyze(state: EmailState):
    print("\n=== Node: Categorizing Email ===")
    prompt = ChatPromptTemplate.from_messages([
        ("system", """Analyze the incoming customer email.
Classify it into one of these categories: 'billing', 'technical', 'feedback', or 'general'.
Also, identify 1-3 tasks that need to be accomplished to resolve this email.
Return the response ONLY as a JSON object with keys 'category' and 'tasks' (a list of string task descriptions).
Example:
{{
  \"category\": \"billing\",
  \"tasks\": [\"Verify subscription payment\", \"Issue invoice copy\"]
}}"""),
        ("human", "Subject: {subject}\nBody: {body}")
    ])
    
    chain = prompt | llm
    response = chain.invoke({"subject": state["subject"], "body": state["body"]})
    
    # Parse the json output
    try:
        res_dict = json.loads(response.content.strip().strip("```json").strip("```"))
        category = res_dict.get("category", "general")
        task_descriptions = res_dict.get("tasks", ["Review customer email"])
    except Exception as e:
        print("Error parsing model response:", e)
        category = "general"
        task_descriptions = ["Review customer email"]
        
    # Convert to structured Task dicts with 'pending' status
    tasks = [Task(description=d, status="pending") for d in task_descriptions]
    
    return {
        "category": category,
        "tasks": tasks
    }

--- OpenAI API connection failed (Error code: 429 - {'error': {'message': 'You exceeded your c...). Falling back to Mock LLM ---


### 3. Specialized Handler Nodes
Each node handles categorization-specific tasks and generates draft replies.

In [4]:
def handle_billing(state: EmailState):
    print("=== Node: Handling Billing Email ===")
    # Update task status
    updated_tasks = []
    for t in state["tasks"]:
        if "verify" in t["description"].lower() or "check" in t["description"].lower():
            # Auto-complete billing checks
            updated_tasks.append(Task(description=t["description"], status="completed"))
        else:
            updated_tasks.append(Task(description=t["description"], status="in_progress"))
            
    draft = f"Dear Customer,\n\nThank you for contacting billing support. We have verified your request and our accounts team is looking into it.\n\nBest regards,\nBilling Team"
    return {
        "tasks": updated_tasks,
        "reply_draft": draft
    }

def handle_technical(state: EmailState):
    print("=== Node: Handling Technical Email ===")
    updated_tasks = [Task(description=t["description"], status="in_progress") for t in state["tasks"]]
    draft = f"Dear Customer,\n\nThank you for reaching out to Technical Support. We have logged your issue and our systems engineers are currently investigating.\n\nBest regards,\nTech Team"
    return {
        "tasks": updated_tasks,
        "reply_draft": draft
    }

def handle_feedback(state: EmailState):
    print("=== Node: Handling Feedback Email ===")
    # We instantly complete feedback log tasks
    updated_tasks = [Task(description=t["description"], status="completed") for t in state["tasks"]]
    draft = f"Dear Customer,\n\nThank you so much for your feedback! We truly value your inputs and have shared them directly with our product design team.\n\nBest regards,\nProduct Team"
    return {
        "tasks": updated_tasks,
        "reply_draft": draft
    }

def handle_general(state: EmailState):
    print("=== Node: Handling General Email ===")
    updated_tasks = [Task(description=t["description"], status="in_progress") for t in state["tasks"]]
    draft = f"Dear Customer,\n\nThank you for contacting customer support. We are reviewing your inquiry and will update you shortly.\n\nBest regards,\nSupport Team"
    return {
        "tasks": updated_tasks,
        "reply_draft": draft
    }

### 4. Human Approval Node
This node is designed as a pass-through node where execution stops (via interrupts). The human agent can review the tasks, make edits to the draft, change task statuses to `completed`, and approve the email.

In [5]:
def human_approval_node(state: EmailState):
    print("=== Node: Human Review Intercept ===")
    # Pass-through logic; actual modifications are made by the human during the interrupt pause
    return {}

### 5. Final Send/Complete Node
This node runs after the human review is completed and the email is approved.

In [6]:
def finalize_and_send(state: EmailState):
    print("=== Node: Finalizing & Sending Email ===")
    if state["approved"]:
        print(f"\n>>> SENDING EMAIL TO {state['sender']} <<<")
        print(state["reply_draft"])
        print(">>> EMAIL SENT <<<")
        
        # Complete all remaining tasks
        completed_tasks = [Task(description=t["description"], status="completed") for t in state["tasks"]]
        return {"tasks": completed_tasks}
    else:
        print("\n>>> Graph Completed without Sending (Draft Discarded/Disapproved) <<<")
        return {}

### 6. Build and Compile Workflow Graph

In [7]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# Routing Function for Conditional Edge
def route_email_by_category(state: EmailState):
    cat = state["category"]
    print(f"Routing category: {cat}")
    if cat == "billing":
        return "billing"
    elif cat == "technical":
        return "technical"
    elif cat == "feedback":
        return "feedback"
    else:
        return "general"

builder = StateGraph(EmailState)

# Add Nodes
builder.add_node("categorize", categorize_and_analyze)
builder.add_node("billing", handle_billing)
builder.add_node("technical", handle_technical)
builder.add_node("feedback", handle_feedback)
builder.add_node("general", handle_general)
builder.add_node("human_approval", human_approval_node)
builder.add_node("send_reply", finalize_and_send)

# Define Edges
builder.add_edge(START, "categorize")

# Conditional edge routing based on classification
builder.add_conditional_edges(
    "categorize",
    route_email_by_category,
    {
        "billing": "billing",
        "technical": "technical",
        "feedback": "feedback",
        "general": "general"
    }
)

# All handlers merge into human approval node
builder.add_edge("billing", "human_approval")
builder.add_edge("technical", "human_approval")
builder.add_edge("feedback", "human_approval")
builder.add_edge("general", "human_approval")

# After approval node, go to send reply node
builder.add_edge("human_approval", "send_reply")
builder.add_edge("send_reply", END)

# Use Memory for checkpointer persistence & Human-in-the-Loop interrupts
memory = MemorySaver()

# We configure graph to pause BEFORE executing human_approval
email_workflow = builder.compile(checkpointer=memory, interrupt_before=["human_approval"])

### 7. Run workflow on a customer email

In [8]:
config = {"configurable": {"thread_id": "email-run-1"}}

sample_email = {
    "email_id": "12345",
    "sender": "customer_billing_help@example.com",
    "subject": "Subscription renewal failed",
    "body": "Hello, I received an email stating that my credit card failed to renew my monthly premium plan subscription. Can you verify my payment status and send me the last invoice? Thank you!",
    "category": "",
    "tasks": [],
    "reply_draft": "",
    "approved": False
}

# Trigger the workflow execution
email_workflow.invoke(sample_email, config)


=== Node: Categorizing Email ===
Routing category: billing
=== Node: Handling Billing Email ===


{'email_id': '12345',
 'sender': 'customer_billing_help@example.com',
 'subject': 'Subscription renewal failed',
 'body': 'Hello, I received an email stating that my credit card failed to renew my monthly premium plan subscription. Can you verify my payment status and send me the last invoice? Thank you!',
 'category': 'billing',
 'tasks': [{'description': 'Verify subscription payment',
   'status': 'completed'},
  {'description': 'Issue invoice copy', 'status': 'in_progress'}],
 'reply_draft': 'Dear Customer,\n\nThank you for contacting billing support. We have verified your request and our accounts team is looking into it.\n\nBest regards,\nBilling Team',
 'approved': False}

### 8. Inspect Paused State (Human Review)

# Retrieve the state details where it was interrupted
current_state = email_workflow.get_state(config)

print("Is paused?")
print("Next node scheduled to run:", current_state.next)
print("\n--- Classified Category ---")
print(current_state.values["category"])

print("\n--- Task Checklist & Statuses ---")
for index, task in enumerate(current_state.values["tasks"]):
    print(f"{index + 1}. [{task['status'].upper()}] {task['description']}")

print("\n--- Drafted Reply ---")
print(current_state.values["reply_draft"])

### 9. Update State (Simulating Human Action)
We'll simulate a human agent modifying the response draft, completing manual checklist items, and approving the final send.

# Update the draft reply and tasks list
customized_draft = current_state.values["reply_draft"] + "\n\nUPDATE: Your failed transaction has been cleared, and I've confirmed that the billing system successfully processed your card manually. The copy of your last invoice is attached!"

# We mark all tasks as completed
updated_tasks = [Task(description=t["description"], status="completed") for t in current_state.values["tasks"]]

email_workflow.update_state(
    config,
    {
        "reply_draft": customized_draft,
        "tasks": updated_tasks,
        "approved": True
    },
    as_node="human_approval"
)
print("State updated successfully with human input!")

### 10. Resume and Finish workflow

In [11]:
print("Resuming workflow execution...")
final_state = email_workflow.invoke(None, config)

print("\n--- Final Workflow Statuses ---")
for index, task in enumerate(final_state["tasks"]):
    print(f"{index + 1}. [{task['status'].upper()}] {task['description']}")

Resuming workflow execution...
=== Node: Finalizing & Sending Email ===

>>> SENDING EMAIL TO customer_billing_help@example.com <<<
Dear Customer,

Thank you for contacting billing support. We have verified your request and our accounts team is looking into it.

Best regards,
Billing Team

UPDATE: Your failed transaction has been cleared, and I've confirmed that the billing system successfully processed your card manually. The copy of your last invoice is attached!
>>> EMAIL SENT <<<

--- Final Workflow Statuses ---
1. [COMPLETED] Verify subscription payment
2. [COMPLETED] Issue invoice copy
